<a href="https://colab.research.google.com/github/mhabibier/scikit-learn-cookbook/blob/main/02_Pre_Model_Workflow_and_Data_Preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Chapter 2 — Pre-Model Workflow and Data Preprocessing

**Muhammad Habibie Rabbani · TK-47-04 · Teknik Komputer, Telkom University**

Reproduksi dan adaptasi resep Chapter 2 dari John Sukup, *scikit-learn Cookbook*, edisi ketiga (Packt, 2025). Contoh resmi buku digunakan sebagai acuan metode. Penjelasan, audit, interpretasi, dan contoh evaluasi ditulis untuk notebook ini. [Kode resmi Chapter 2](https://github.com/PacktPublishing/scikit-learn-Cookbook-Third-Edition/tree/main/Chapter_2).

## Tujuan Pembelajaran

1. Mengaudit kualitas data sebelum membangun model.
2. Membandingkan mean, KNN, dan iterative imputation.
3. Membedakan standardisasi, MinMax scaling, dan normalisasi baris.
4. Mengodekan kategori dan menggabungkan transformasi berdasarkan jenis kolom.
5. Mencegah data leakage dengan train/test split dan `Pipeline`.
6. Membuat feature baru dan memilih feature pada data training.
7. Mengevaluasi pipeline regresi dengan baseline yang masuk akal.

## 1. Lingkungan dan Reproducibility

Contoh menggunakan random seed 2024 seperti notebook buku. Versi Python dan library dicatat agar perbedaan output dapat ditelusuri. Data sintetis hanya untuk demonstrasi; kesimpulan tentang populasi nyata tidak dapat ditarik darinya.

In [1]:
import sys
import numpy as np
import pandas as pd
import sklearn

RANDOM_STATE = 2024

print("Python:", sys.version.split()[0])
print("NumPy:", np.__version__)
print("pandas:", pd.__version__)
print("scikit-learn:", sklearn.__version__)

Python: 3.12.14
NumPy: 2.3.5
pandas: 2.2.3
scikit-learn: 1.8.0


## 2. Raw Data dan Audit Kualitas

Data dengan nilai hilang, satuan yang berbeda, atau kategori berbentuk teks perlu ditangani sesuai konteks. Sebelum memilih metode, periksa ukuran, tipe kolom, nilai hilang, duplikasi, dan nilai tak hingga. Pada contoh buku, 20 baris × 10 feature numerik dibuat secara acak dan sekitar 20% nilai dihilangkan.

In [2]:
rng_numeric = np.random.RandomState(RANDOM_STATE)
n_samples, n_features = 20, 10
df = pd.DataFrame({
    f"Feature{i+1}": rng_numeric.uniform(0, 100, n_samples)
    for i in range(n_features)
})
for column in df.columns:
    df.loc[rng_numeric.random(n_samples) < 0.20, column] = np.nan

numeric_audit = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "missing": df.isna().sum(),
    "missing_pct": (100 * df.isna().mean()).round(1),
    "unique_non_missing": df.nunique(dropna=True),
})
print("Shape:", df.shape)
print("Exact duplicate rows:", int(df.duplicated().sum()))
print("Infinite values:", int(np.isinf(df.to_numpy()).sum()))
print("Missing values:", int(df.isna().sum().sum()))
print(numeric_audit.to_string())
print("\nFirst five rows:\n", df.head().to_string())

Shape: (20, 10)
Exact duplicate rows: 0
Infinite values: 0
Missing values: 44
             dtype  missing  missing_pct  unique_non_missing
Feature1   float64        3         15.0                  17
Feature2   float64        5         25.0                  15
Feature3   float64        4         20.0                  16
Feature4   float64        4         20.0                  16
Feature5   float64        5         25.0                  15
Feature6   float64        5         25.0                  15
Feature7   float64        7         35.0                  13
Feature8   float64        6         30.0                  14
Feature9   float64        2         10.0                  18
Feature10  float64        3         15.0                  17

First five rows:
     Feature1   Feature2   Feature3   Feature4   Feature5   Feature6   Feature7   Feature8   Feature9  Feature10
0  58.801452        NaN  42.009814  34.680397  49.962259        NaN        NaN  89.954588  41.152421        NaN
1  69.91

### Interpretasi Audit

Baris duplikat persis dan nilai yang berulang mempunyai arti berbeda. Dataset buatan ini tidak memiliki identitas subjek, sehingga jumlah nilai yang sama tidak dapat dipakai untuk menyatakan ada orang duplikat. Nilai hilang dimasukkan secara acak hanya untuk latihan. Tidak ada bukti bahwa mekanisme missing pada data nyata juga acak. Bila suatu kolom seluruhnya kosong atau nilai ekstrem berasal dari kesalahan pencatatan, keputusan perbaikan perlu dibuat sebelum modeling.

## 3. Handling Missing Data

`SimpleImputer(strategy="mean")` memakai rerata setiap kolom. `KNNImputer` memperkirakan nilai dari tetangga yang mirip pada feature teramati; perbedaan skala dapat memengaruhi jaraknya. `IterativeImputer` memperkirakan suatu feature dari feature lain secara bergantian. Ketiganya tidak dapat membuktikan nilai asli yang hilang. Pada contoh kecil ini, semua imputer di-*fit* pada data yang sama semata untuk membandingkan metode; pada pekerjaan prediktif imputer harus dilatih hanya pada data training.

In [3]:
from sklearn.impute import SimpleImputer

mean_imputer = SimpleImputer(strategy="mean")
mean_imputed_df = pd.DataFrame(
    mean_imputer.fit_transform(df), columns=df.columns, index=df.index
)
print("Mean-imputed first five rows:\n", mean_imputed_df.head().round(2).to_string())
print("Remaining missing:", int(mean_imputed_df.isna().sum().sum()))

Mean-imputed first five rows:
    Feature1  Feature2  Feature3  Feature4  Feature5  Feature6  Feature7  Feature8  Feature9  Feature10
0     58.80     53.86     42.01     34.68     49.96     57.82     51.15     89.95     41.15      48.40
1     69.91      9.55      6.44     31.29     37.97     57.82     82.01     50.72     75.80      91.38
2     52.56     96.09     59.64     84.71     46.06     84.11     51.15     70.29      1.78       4.12
3     52.56     25.18     83.73     88.02     16.89     97.21     84.70     50.72     60.62      80.08
4     20.50     53.86     89.25     67.66     58.64     78.23     60.91     50.72     65.11      99.12
Remaining missing: 0


In [4]:
from sklearn.impute import KNNImputer

knn_imputer = KNNImputer(n_neighbors=2)
knn_imputed_df = pd.DataFrame(
    knn_imputer.fit_transform(df), columns=df.columns, index=df.index
)
print("KNN-imputed first five rows:\n", knn_imputed_df.head().round(2).to_string())
print("Remaining missing:", int(knn_imputed_df.isna().sum().sum()))

KNN-imputed first five rows:
    Feature1  Feature2  Feature3  Feature4  Feature5  Feature6  Feature7  Feature8  Feature9  Feature10
0     58.80     48.95     42.01     34.68     49.96     93.91     52.55     89.95     41.15      59.83
1     69.91      9.55      6.44     31.29     37.97     86.79     82.01     45.42     75.80      91.38
2     67.75     96.09     59.64     84.71     73.34     84.11     59.01     70.29      1.78       4.12
3     47.88     25.18     83.73     88.02     16.89     97.21     84.70     64.51     39.73      80.08
4     20.50     31.67     89.25     67.66     58.64     78.23     60.91     25.90     65.11      99.12
Remaining missing: 0


In [5]:
from sklearn.experimental import enable_iterative_imputer  # noqa: F401
from sklearn.impute import IterativeImputer

iterative_imputer = IterativeImputer(random_state=RANDOM_STATE, max_iter=20)
iterative_imputed_df = pd.DataFrame(
    iterative_imputer.fit_transform(df), columns=df.columns, index=df.index
)
print("Iterative-imputed first five rows:\n", iterative_imputed_df.head().round(2).to_string())
print("Remaining missing:", int(iterative_imputed_df.isna().sum().sum()))

Iterative-imputed first five rows:
    Feature1  Feature2  Feature3  Feature4  Feature5  Feature6  Feature7  Feature8  Feature9  Feature10
0     58.80     54.37     42.01     34.68     49.96     58.68     51.18     89.95     41.15      48.29
1     69.91      9.55      6.44     31.29     37.97     60.07     82.01     50.71     75.80      91.38
2     52.54     96.09     59.64     84.71     46.12     84.11     51.16     70.29      1.78       4.12
3     52.63     25.18     83.73     88.02     16.89     97.21     84.70     50.68     50.65      80.08
4     20.50     53.20     89.25     67.66     58.64     78.23     60.91     50.69     65.11      99.12
Remaining missing: 0


In [6]:
first_missing_row, first_missing_col = np.argwhere(df.isna().to_numpy())[0]
feature_name = df.columns[first_missing_col]
print("Example missing cell: row", first_missing_row, "column", feature_name)
print("Mean estimate:", round(mean_imputed_df.iloc[first_missing_row, first_missing_col], 2))
print("KNN estimate:", round(knn_imputed_df.iloc[first_missing_row, first_missing_col], 2))
print("Iterative estimate:", round(iterative_imputed_df.iloc[first_missing_row, first_missing_col], 2))
print("All output shapes:", [x.shape for x in
    (mean_imputed_df, knn_imputed_df, iterative_imputed_df)])

Example missing cell: row 0 column Feature2
Mean estimate: 53.86
KNN estimate: 48.95
Iterative estimate: 54.37
All output shapes: [(20, 10), (20, 10), (20, 10)]


### Analisis Imputation

Ketiga metode menghasilkan tabel 20 × 10 tanpa nilai hilang, tetapi nilai taksirannya dapat berbeda. Mean cepat dan sederhana, namun meratakan variasi; KNN peka terhadap definisi jarak dan skala; iterative imputation memakai hubungan antarkolom yang pada 20 baris sintetis dapat bersifat kebetulan. Angka yang diisi bukan ground truth. Hindari memilih metode hanya dari tampilan hasil; validasi menggunakan data relevan dan tujuan prediksi.

## 4. Scaling Techniques

Semua demonstrasi berikut memakai hasil imputasi iterative agar tidak ada NaN. **StandardScaler** menghasilkan rerata sekitar 0 dan simpangan baku sekitar 1 per kolom: $z=(x-\mu)/\sigma$. **MinMaxScaler** memetakan nilai training ke interval 0–1: $(x-\min)/(\max-\min)$. **Normalizer** membagi tiap baris dengan normanya, sehingga panjang vektor baris mendekati 1. Standardisasi tidak membuat distribusi otomatis menjadi normal dan tidak menghilangkan outlier.

In [7]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler, Normalizer

standard_scaler = StandardScaler()
standardized_df = pd.DataFrame(
    standard_scaler.fit_transform(iterative_imputed_df),
    columns=df.columns, index=df.index
)
print("StandardScaler first rows:\n", standardized_df.head(3).round(3).to_string())
print("Column means:", standardized_df.mean().round(6).to_dict())
print("Column population std:", standardized_df.std(ddof=0).round(6).to_dict())

StandardScaler first rows:
    Feature1  Feature2  Feature3  Feature4  Feature5  Feature6  Feature7  Feature8  Feature9  Feature10
0     0.272     0.019    -0.373    -0.894     0.189     0.032     0.002     1.684    -0.781     -0.004
1     0.756    -1.875    -1.592    -1.044    -0.392     0.086     1.427    -0.001     0.652      1.429
2    -0.001     1.784     0.231     1.312     0.003     1.022     0.001     0.840    -2.410     -1.472
Column means: {'Feature1': -0.0, 'Feature2': 0.0, 'Feature3': 0.0, 'Feature4': -0.0, 'Feature5': 0.0, 'Feature6': -0.0, 'Feature7': -0.0, 'Feature8': 0.0, 'Feature9': 0.0, 'Feature10': -0.0}
Column population std: {'Feature1': 1.0, 'Feature2': 1.0, 'Feature3': 1.0, 'Feature4': 1.0, 'Feature5': 1.0, 'Feature6': 1.0, 'Feature7': 1.0, 'Feature8': 1.0, 'Feature9': 1.0, 'Feature10': 1.0}


In [8]:
minmax_scaler = MinMaxScaler()
minmax_df = pd.DataFrame(
    minmax_scaler.fit_transform(iterative_imputed_df),
    columns=df.columns, index=df.index
)
print("MinMaxScaler first rows:\n", minmax_df.head(3).round(3).to_string())
print("Smallest overall value:", round(minmax_df.min().min(), 6))
print("Largest overall value:", round(minmax_df.max().max(), 6))

MinMaxScaler first rows:
    Feature1  Feature2  Feature3  Feature4  Feature5  Feature6  Feature7  Feature8  Feature9  Feature10
0     0.604     0.518     0.381     0.355     0.569     0.556     0.435     0.963     0.402      0.465
1     0.721     0.000     0.000     0.314     0.413     0.572     0.833     0.539     0.756      0.919
2     0.537     1.000     0.569     0.960     0.519     0.849     0.435     0.750     0.000      0.000
Smallest overall value: 0.0
Largest overall value: 1.0


In [9]:
row_normalizer = Normalizer()
normalized_df = pd.DataFrame(
    row_normalizer.fit_transform(iterative_imputed_df),
    columns=df.columns, index=df.index
)
print("Normalizer first rows:\n", normalized_df.head(3).round(3).to_string())
print("First five row norms:",
      np.linalg.norm(normalized_df.to_numpy(), axis=1)[:5].round(6))

Normalizer first rows:
    Feature1  Feature2  Feature3  Feature4  Feature5  Feature6  Feature7  Feature8  Feature9  Feature10
0     0.339     0.314     0.242     0.200     0.288     0.338     0.295     0.519     0.237      0.279
1     0.377     0.051     0.035     0.169     0.205     0.324     0.442     0.273     0.408      0.492
2     0.264     0.483     0.300     0.426     0.232     0.423     0.257     0.354     0.009      0.021
First five row norms: [1. 1. 1. 1. 1.]


### Analisis Scaling

StandardScaler dan MinMaxScaler belajar statistik **per kolom** dari data yang diberikan. Normalizer bekerja **per baris**. Nilai MinMax pada data test dapat berada di luar 0–1 jika melampaui rentang training. Pada model berbasis jarak atau koefisien, skala sering penting; untuk pohon keputusan, scaling biasanya tidak diperlukan. Statistik scaler pada model evaluasi nanti dipelajari hanya dari training melalui pipeline.

## 5. Encoding Categorical Variables

`OneHotEncoder` membuat kolom indikator untuk kategori nominal. `LabelEncoder` dirancang untuk label target `y`; memberi kode 0, 1, 2 pada feature nominal dapat menimbulkan kesan urutan numerik yang tidak ada. Contoh resmi mengaplikasikan `LabelEncoder` pada feature, sehingga di sini penggunaannya ditunjukkan pada target. `ColumnTransformer` menerapkan langkah berbeda pada kolom numerik dan kategori.

In [10]:
rng_cat = np.random.RandomState(RANDOM_STATE)
categorical_data = pd.DataFrame({
    "Department": rng_cat.choice(["A", "B", "C", "D"], size=20),
    "Position": rng_cat.choice(["Junior", "Senior", "Manager"], size=20),
    "Location": rng_cat.choice(["NY", "SF", "LA", "CHI"], size=20),
})
print("Category counts by column:")
for col in categorical_data:
    print(col, categorical_data[col].value_counts().to_dict())
print(categorical_data.head().to_string(index=False))

Category counts by column:
Department {'A': 7, 'C': 5, 'D': 4, 'B': 4}
Position {'Manager': 11, 'Senior': 6, 'Junior': 3}
Location {'LA': 7, 'NY': 6, 'SF': 4, 'CHI': 3}
Department Position Location
         A  Manager       LA
         C  Manager       LA
         A   Junior       NY
         A  Manager       LA
         D  Manager       NY


In [11]:
from sklearn.preprocessing import OneHotEncoder, LabelEncoder

onehot_encoder = OneHotEncoder(sparse_output=False, handle_unknown="ignore")
onehot_df = pd.DataFrame(
    onehot_encoder.fit_transform(categorical_data),
    columns=onehot_encoder.get_feature_names_out(categorical_data.columns),
    index=categorical_data.index,
)
print("One-hot shape:", onehot_df.shape)
print(onehot_df.head().to_string(index=False))
print("Unseen category, same number of columns:",
      onehot_encoder.transform(pd.DataFrame({
          "Department": ["Unknown"], "Position": ["Junior"], "Location": ["NY"]
      })).shape)

One-hot shape: (20, 11)
 Department_A  Department_B  Department_C  Department_D  Position_Junior  Position_Manager  Position_Senior  Location_CHI  Location_LA  Location_NY  Location_SF
          1.0           0.0           0.0           0.0              0.0               1.0              0.0           0.0          1.0          0.0          0.0
          0.0           0.0           1.0           0.0              0.0               1.0              0.0           0.0          1.0          0.0          0.0
          1.0           0.0           0.0           0.0              1.0               0.0              0.0           0.0          0.0          1.0          0.0
          1.0           0.0           0.0           0.0              0.0               1.0              0.0           0.0          1.0          0.0          0.0
          0.0           0.0           0.0           1.0              0.0               1.0              0.0           0.0          0.0          1.0          0.0
Unseen cat

In [12]:
target_labels = categorical_data["Department"]
label_encoder = LabelEncoder()
encoded_target = label_encoder.fit_transform(target_labels)
print("Target classes:", label_encoder.classes_.tolist())
print("First ten encoded target values:", encoded_target[:10].tolist())
print("Decoded first ten:", label_encoder.inverse_transform(encoded_target[:10]).tolist())

Target classes: ['A', 'B', 'C', 'D']
First ten encoded target values: [0, 2, 0, 0, 3, 0, 2, 3, 1, 3]
Decoded first ten: ['A', 'C', 'A', 'A', 'D', 'A', 'C', 'D', 'B', 'D']


### Analisis Encoding

Keluaran one-hot menambah jumlah feature tanpa memberi jarak numerik palsu antarkategori. `handle_unknown="ignore"` mempertahankan bentuk data ketika kategori baru muncul, dengan indikator nol untuk kategori yang tidak dikenali. Label target yang diubah oleh `LabelEncoder` dapat dikembalikan memakai `inverse_transform()`. Dalam aplikasi nyata, pemilihan kategori dan penanganan kategori langka perlu disesuaikan dengan data training.

In [13]:
from sklearn.compose import ColumnTransformer

rng_mixed = np.random.RandomState(RANDOM_STATE)
mixed_data = pd.DataFrame({
    "Age": rng_mixed.randint(25, 65, size=20),
    "Salary": np.round(rng_mixed.normal(60000, 15000, size=20), 2),
    "Experience": rng_mixed.randint(1, 20, size=20),
    "Department": rng_mixed.choice(["IT", "HR", "Sales", "Finance"], size=20),
    "Position": rng_mixed.choice(["Junior", "Senior", "Manager"], size=20),
})
numeric_columns = ["Age", "Salary", "Experience"]
category_columns = ["Department", "Position"]
column_transformer = ColumnTransformer([
    ("num", StandardScaler(), numeric_columns),
    ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False),
     category_columns),
])
mixed_transformed = column_transformer.fit_transform(mixed_data)
mixed_transformed_df = pd.DataFrame(
    mixed_transformed,
    columns=column_transformer.get_feature_names_out(),
    index=mixed_data.index,
)
print("Mixed input shape:", mixed_data.shape)
print("Transformed shape:", mixed_transformed_df.shape)
print("Output features:", mixed_transformed_df.columns.tolist())
print(mixed_transformed_df.head(3).round(3).to_string(index=False))

Mixed input shape: (20, 5)
Transformed shape: (20, 10)
Output features: ['num__Age', 'num__Salary', 'num__Experience', 'cat__Department_Finance', 'cat__Department_HR', 'cat__Department_IT', 'cat__Department_Sales', 'cat__Position_Junior', 'cat__Position_Manager', 'cat__Position_Senior']
 num__Age  num__Salary  num__Experience  cat__Department_Finance  cat__Department_HR  cat__Department_IT  cat__Department_Sales  cat__Position_Junior  cat__Position_Manager  cat__Position_Senior
   -1.045       -0.305            0.327                      1.0                 0.0                 0.0                    0.0                   0.0                    1.0                   0.0
    0.728        1.105            1.262                      0.0                 0.0                 0.0                    1.0                   1.0                    0.0                   0.0
   -1.636       -1.582            1.075                      1.0                 0.0                 0.0                    0.0

### Analisis ColumnTransformer

Tiga kolom numerik distandardisasi, sedangkan dua kolom kategori diubah menjadi indikator. Nama hasil dari `get_feature_names_out()` mempertahankan kaitan dengan transformer asal. Ini demonstrasi transformasi saja, tanpa target prediksi. Mengambil kolom terakhir hasil one-hot sebagai `y` lalu memakai indikator kategori lain sebagai feature akan membuat tugas prediksi bocor, sehingga bagian modeling berikut menggunakan target asli dari dataset regresi.

## 6. Train/Test Split dan Pipeline

Untuk memperlihatkan urutan preprocessing yang benar, bagian ini memakai dataset diabetes bawaan scikit-learn, berisi target regresi yang terpisah dari sepuluh feature. Nilai hilang **ditambahkan secara artifisial** pada sebagian feature training dan test. Dataset ini digunakan untuk menguji API, bukan untuk keputusan medis. Pemisahan dilakukan sebelum imputer atau scaler di-*fit*.

In [14]:
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split

diabetes = load_diabetes(as_frame=True)
X_diabetes, y_diabetes = diabetes.data, diabetes.target
X_train, X_test, y_train, y_test = train_test_split(
    X_diabetes, y_diabetes, test_size=0.2, random_state=RANDOM_STATE
)
rng_missing = np.random.default_rng(RANDOM_STATE)
X_train_missing = X_train.copy()
X_test_missing = X_test.copy()
for data_split in (X_train_missing, X_test_missing):
    mask = rng_missing.random(data_split.shape) < 0.05
    data_split.iloc[:, :] = data_split.mask(mask)

print("Training:", X_train_missing.shape, "missing:",
      int(X_train_missing.isna().sum().sum()))
print("Test:", X_test_missing.shape, "missing:",
      int(X_test_missing.isna().sum().sum()))
print("Target type:", y_train.dtype, "range:",
      (float(y_train.min()), float(y_train.max())))
print("Any entirely missing training feature:",
      bool(X_train_missing.isna().all().any()))

Training: (353, 10) missing: 178
Test: (89, 10) missing: 48
Target type: float64 range: (25.0, 346.0)
Any entirely missing training feature: False


In [15]:
from sklearn.pipeline import Pipeline

numeric_preprocessor = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])
X_train_transformed = pd.DataFrame(
    numeric_preprocessor.fit_transform(X_train_missing),
    columns=X_train.columns, index=X_train.index,
)
X_test_transformed = pd.DataFrame(
    numeric_preprocessor.transform(X_test_missing),
    columns=X_test.columns, index=X_test.index,
)
print("Training transformed:", X_train_transformed.shape)
print("Test transformed:", X_test_transformed.shape)
print("Remaining missing values:",
      int(X_train_transformed.isna().sum().sum() +
          X_test_transformed.isna().sum().sum()))
print("First five training column means:",
      X_train_transformed.mean().head().round(5).to_dict())

Training transformed: (353, 10)
Test transformed: (89, 10)
Remaining missing values: 0
First five training column means: {'age': 0.0, 'sex': 0.0, 'bmi': 0.0, 'bp': 0.0, 's1': 0.0}


### Analisis Pipeline Preprocessing

`fit_transform()` hanya dipanggil pada training. `transform()` pada test memakai median dan parameter scaler yang dipelajari dari training. Rerata kolom training setelah standardisasi mendekati nol; rerata test tidak harus nol. Data diabetes aslinya telah dinormalisasi oleh pembuat dataset, sehingga scaling tambahan di sini menunjukkan cara kerja pipeline dan bukan klaim peningkatan akurasi.

In [16]:
from sklearn.linear_model import Ridge
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import KFold, cross_val_score

model_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", Ridge(alpha=1.0)),
])
cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_r2 = cross_val_score(
    model_pipeline, X_train_missing, y_train, cv=cv, scoring="r2"
)
model_pipeline.fit(X_train_missing, y_train)
test_prediction = model_pipeline.predict(X_test_missing)
baseline = DummyRegressor(strategy="mean").fit(X_train_missing, y_train)
baseline_prediction = baseline.predict(X_test_missing)

print("CV R² mean ± std:", round(cv_r2.mean(), 3), "±", round(cv_r2.std(), 3))
print("Pipeline test R²:", round(r2_score(y_test, test_prediction), 3))
print("Pipeline test MAE:", round(mean_absolute_error(y_test, test_prediction), 3))
print("Mean baseline test R²:", round(r2_score(y_test, baseline_prediction), 3))
print("Mean baseline test MAE:",
      round(mean_absolute_error(y_test, baseline_prediction), 3))

CV R² mean ± std: 0.477 ± 0.12
Pipeline test R²: 0.444
Pipeline test MAE: 47.986
Mean baseline test R²: -0.0
Mean baseline test MAE: 68.171


In [17]:
from sklearn import set_config

set_config(display="diagram")
try:
    from IPython.display import display
    display(model_pipeline)
except ImportError:
    print(model_pipeline)
print("Pipeline steps:", [name for name, _ in model_pipeline.steps])

Pipeline(steps=[('imputer', SimpleImputer(strategy='median')),
                ('scaler', StandardScaler()), ('model', Ridge())])
Pipeline steps: ['imputer', 'scaler', 'model']


### Analisis Evaluasi dan Visualisasi Pipeline

`cross_val_score()` menerima **pipeline lengkap**, sehingga median dan skala dihitung ulang di tiap lipatan training. Skor test digunakan setelah konfigurasi model ditetapkan. R² dibandingkan dengan baseline yang selalu menebak mean target, sedangkan MAE menunjukkan kesalahan dalam satuan target. Diagram pipeline membantu memeriksa urutan imputer → scaler → model. Dataset ini kecil dan missing value buatan; hasil tidak dapat digeneralisasikan sebagai performa klinis.

## 7. Feature Engineering dan Selection

`PolynomialFeatures` menambah kuadrat dan interaksi, `KBinsDiscretizer` mengubah nilai kontinu menjadi beberapa bin, `RFE` menghapus feature berdasarkan estimator secara berulang, dan `SelectFromModel` memilih feature berdasarkan besarnya koefisien. Setiap transformer dan selector berikut di-*fit* hanya pada training. Pilihan akhir untuk model sebaiknya diuji melalui cross-validation dengan transformer berada dalam pipeline.

In [18]:
from sklearn.preprocessing import PolynomialFeatures

poly = PolynomialFeatures(degree=2, include_bias=False)
X_train_poly = poly.fit_transform(X_train_transformed)
print("Original feature count:", X_train_transformed.shape[1])
print("Polynomial feature count:", X_train_poly.shape[1])
print("First 15 feature names:",
      poly.get_feature_names_out(X_train_transformed.columns)[:15].tolist())
print("Test transform shape:", poly.transform(X_test_transformed).shape)

Original feature count: 10
Polynomial feature count: 65
First 15 feature names: ['age', 'sex', 'bmi', 'bp', 's1', 's2', 's3', 's4', 's5', 's6', 'age^2', 'age sex', 'age bmi', 'age bp', 'age s1']
Test transform shape: (89, 65)


In [19]:
from sklearn.preprocessing import KBinsDiscretizer

kbins = KBinsDiscretizer(n_bins=3, encode="ordinal", strategy="uniform")
X_train_binned = kbins.fit_transform(X_train_transformed)
X_test_binned = kbins.transform(X_test_transformed)
print("Bins per feature:", kbins.n_bins_.tolist())
print("First three binned rows:\n", X_train_binned[:3].astype(int))
print("Binned test shape:", X_test_binned.shape)

Bins per feature: [3, 3, 3, 3, 3, 3, 3, 3, 3, 3]
First three binned rows:
 [[1 2 1 1 1 1 0 1 1 1]
 [2 0 1 1 1 1 0 1 1 1]
 [0 2 0 2 2 2 0 2 1 1]]
Binned test shape: (89, 10)


In [20]:
from sklearn.feature_selection import RFE, SelectFromModel
from sklearn.linear_model import LinearRegression

rfe = RFE(estimator=LinearRegression(), n_features_to_select=5)
rfe.fit(X_train_transformed, y_train)
print("RFE selected:", X_train_transformed.columns[rfe.support_].tolist())
print("RFE rankings:", dict(zip(X_train_transformed.columns, rfe.ranking_)))
print("RFE test shape:", rfe.transform(X_test_transformed).shape)

selector = SelectFromModel(estimator=LinearRegression(), threshold="mean")
selector.fit(X_train_transformed, y_train)
selected_report = pd.DataFrame({
    "feature": X_train_transformed.columns,
    "abs_coefficient": np.abs(selector.estimator_.coef_),
    "selected": selector.get_support(),
}).sort_values("abs_coefficient", ascending=False)
print("SelectFromModel threshold:", round(selector.threshold_, 3))
print(selected_report.round(3).to_string(index=False))
print("Selected test shape:", selector.transform(X_test_transformed).shape)

RFE selected: ['bmi', 'bp', 's1', 's2', 's5']
RFE rankings: {'age': np.int64(6), 'sex': np.int64(2), 'bmi': np.int64(1), 'bp': np.int64(1), 's1': np.int64(1), 's2': np.int64(1), 's3': np.int64(4), 's4': np.int64(3), 's5': np.int64(1), 's6': np.int64(5)}
RFE test shape: (89, 5)
SelectFromModel threshold: 12.119
feature  abs_coefficient  selected
     s5           27.496      True
    bmi           26.492      True
     s1           21.471      True
     bp           18.094      True
    sex            9.869     False
     s2            7.335     False
     s4            6.736     False
     s3            1.769     False
     s6            1.425     False
    age            0.500     False
Selected test shape: (89, 4)


### Analisis Feature Engineering

Degree 2 menghasilkan lebih banyak kolom daripada sepuluh feature awal; penambahan ini dapat membantu hubungan nonlinier tetapi juga menambah risiko overfitting. Bin ordinal 0, 1, 2 menyiratkan urutan; untuk kategori interval tanpa asumsi linear, pertimbangkan one-hot encoding. RFE dan SelectFromModel dapat memilih himpunan berbeda karena aturan pemilihannya berbeda. Besar koefisien paling mudah dibandingkan ketika feature telah berada pada skala yang sebanding. Pemilihan feature pada seluruh dataset sebelum cross-validation dapat membocorkan informasi validation.

## 8. Practical Exercise: Comprehensive Pipeline

Solusi latihan resmi memakai California Housing, median imputation, StandardScaler, dan RandomForestRegressor. Notebook mencoba mengambil California Housing. Apabila unduhan tidak tersedia, dataset diabetes bawaan dipakai agar seluruh sel tetap dapat dijalankan secara offline. Nama dataset yang benar-benar dipakai dicetak pada output. Dua dataset memiliki target dan skala berbeda, sehingga angkanya tidak boleh dibandingkan langsung. Scaling dipertahankan untuk meniru urutan buku, walaupun pohon acak pada dasarnya tidak memerlukan standardisasi.

In [21]:
import os
import tempfile
from pathlib import Path
from sklearn.datasets import fetch_california_housing
from sklearn.ensemble import RandomForestRegressor

try:
    if os.environ.get("SCIKIT_COOKBOOK_OFFLINE") == "1":
        raise OSError("offline mode selected")
    exercise_data = fetch_california_housing(
        as_frame=True, data_home=Path(tempfile.gettempdir()) / "scikit-cookbook-housing"
    )
    exercise_name = "California Housing"
except (OSError, ConnectionError) as exc:
    exercise_data = load_diabetes(as_frame=True)
    exercise_name = "Diabetes (offline fallback)"
    print("California Housing unavailable:", type(exc).__name__)

X_exercise, y_exercise = exercise_data.data, exercise_data.target
X_ex_train, X_ex_test, y_ex_train, y_ex_test = train_test_split(
    X_exercise, y_exercise, test_size=0.2, random_state=RANDOM_STATE
)
comprehensive_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", RandomForestRegressor(
        n_estimators=60, max_depth=12, random_state=RANDOM_STATE, n_jobs=-1
    )),
])
comprehensive_pipeline.fit(X_ex_train, y_ex_train)
exercise_prediction = comprehensive_pipeline.predict(X_ex_test)
exercise_baseline = DummyRegressor(strategy="mean").fit(X_ex_train, y_ex_train)
baseline_ex_prediction = exercise_baseline.predict(X_ex_test)
print("Dataset:", exercise_name, "shape:", X_exercise.shape)
print("Random forest test R²:", round(r2_score(y_ex_test, exercise_prediction), 3))
print("Random forest test MAE:",
      round(mean_absolute_error(y_ex_test, exercise_prediction), 3))
print("Baseline test R²:", round(r2_score(y_ex_test, baseline_ex_prediction), 3))
print("Baseline test MAE:",
      round(mean_absolute_error(y_ex_test, baseline_ex_prediction), 3))

Dataset: California Housing shape: (20640, 8)
Random forest test R²: 0.792
Random forest test MAE: 0.346
Baseline test R²: -0.0
Baseline test MAE: 0.921


### Analisis Latihan

Skor R² dan MAE pada output berlaku **hanya untuk dataset yang tercetak di atas**. Baseline mean memberi patokan awal; R² yang lebih besar dan MAE yang lebih kecil menunjukkan kinerja relatif lebih baik pada pembagian test ini. Latihan ini belum melakukan tuning model, pengujian ketahanan pada periode lain, atau audit fairness. Apabila California Housing berhasil diunduh, data tersebut merupakan dataset perumahan historis dan tidak mewakili semua lokasi serta periode.

## 9. Ringkasan Metode

| Metode | Yang dipelajari saat `fit()` | Penggunaan | Batasan utama |
|---|---|---|---|
| SimpleImputer | Statistik tiap kolom | Mengisi NaN | Variasi dapat menyusut |
| KNNImputer | Tetangga berdasarkan data teramati | Mengisi NaN | Peka pada skala dan jarak |
| IterativeImputer | Hubungan antarkolom | Mengisi NaN | Hubungan pada data kecil belum tentu stabil |
| StandardScaler | Mean dan simpangan baku | Menyetarakan skala kolom | Outlier tetap ada |
| MinMaxScaler | Minimum dan maksimum | Memetakan rentang training | Test bisa di luar 0–1 |
| Normalizer | Tidak mempelajari statistik kolom | Menormalkan panjang baris | Mengubah besar relatif observasi |
| OneHotEncoder | Kategori training | Feature nominal | Menambah jumlah kolom |
| LabelEncoder | Kelas target | Label `y` | Tidak untuk feature nominal tanpa alasan |
| ColumnTransformer | Parameter tiap jenis kolom | Data campuran | Daftar kolom harus tepat |
| Pipeline | Tiap langkah pada training | Alur konsisten dan CV | Urutan langkah harus dipilih dengan benar |
| PolynomialFeatures | Kombinasi nama feature | Interaksi dan kuadrat | Feature bertambah cepat |
| KBinsDiscretizer | Batas bin training | Diskretisasi | Informasi kontinu hilang |
| RFE / SelectFromModel | Pentingnya feature pada training | Seleksi feature | Wajib masuk CV untuk evaluasi yang adil |

## 10. Kesimpulan

Chapter ini menunjukkan bahwa kualitas input dan urutan preprocessing memengaruhi keandalan eksperimen ML. Tiga teknik imputation memberi hasil berbeda dan tidak ada yang dapat memastikan nilai asli yang hilang. StandardScaler, MinMaxScaler, dan Normalizer melakukan operasi yang berbeda. One-hot encoding menjaga kategori nominal tanpa membuat jarak numerik semu.

`ColumnTransformer` mengelola data numerik dan kategori, sementara `Pipeline` memastikan imputer dan scaler dilatih hanya pada data training, termasuk di setiap lipatan cross-validation. Feature engineering dapat meningkatkan kemampuan representasi namun menambah kompleksitas; seleksinya perlu divalidasi tanpa memakai informasi test. Evaluasi model harus dibandingkan dengan baseline, serta selalu menyebut dataset, target, dan metrik yang digunakan.

Dataset awal bersifat sintetis dan latihan regresi memiliki keterbatasan konteks. Tidak ada kesimpulan klinis atau kebijakan perumahan yang dapat ditarik dari demonstrasi ini.

## 11. References

1. J. Sukup, *scikit-learn Cookbook: Over 80 Recipes for Machine Learning in Python with scikit-learn*, 3rd ed., Packt Publishing, 2025, ch. 2.
2. Packt Publishing, [official Chapter 2 notebook and exercise solution](https://github.com/PacktPublishing/scikit-learn-Cookbook-Third-Edition/tree/main/Chapter_2).
3. scikit-learn Developers, [Common pitfalls and recommended practices](https://scikit-learn.org/stable/common_pitfalls.html).
4. scikit-learn Developers, [Preprocessing data](https://scikit-learn.org/stable/modules/preprocessing.html) and [Imputation of missing values](https://scikit-learn.org/stable/modules/impute.html).

**Adaptasi dari contoh buku:** target pipeline menggunakan dataset regresi dengan `y` asli, bukan kolom indikator one-hot; `LabelEncoder` diterapkan pada target; pemisahan data dilakukan sebelum fitting preprocessing. Latihan akhir mencoba California Housing dan menyediakan fallback offline. Penjelasan dan interpretasi ditulis ulang untuk pembelajaran.